In [79]:
import os
import cv2
import time
import glob
import copy
import math
import torch
import numpy as np
from PIL import Image
import xgboost as xgb
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import torchvision.models as models
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score
import torchvision.transforms as transforms
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Extração de Características

## Caminhos das imagens

In [ ]:
PASTA_ANOTADAS = ""
PASTA_NAO_ANOTADAS = ""

In [4]:
def create_paths(root, is_labeled = True):
    path = glob.glob(os.path.join(root, "RGB", "*.png"))

    path_data = []
    labels = []

    for p_rgb in path:
        file_name = os.path.basename(p_rgb)
        base_name = os.path.splitext(file_name)[0]

        if is_labeled:
            parts = base_name.split("_")
            label = parts[-1]
            base_name = "_".join(parts[:-1])
            p_hiper = os.path.join(root, "HIPER", base_name + ".npy")
        else:
            p_rx = os.path.join(root, "RX", file_name)
            p_hiper = os.path.join(root, "HIPER", base_name[:-1] + ".npy")

        p_rx = os.path.join(root, "RX", file_name)

        if os.path.exists(p_rx) and os.path.exists(p_hiper):
            path_data.append({
                "rgb": p_rgb,
                "rx": p_rx,
                "hiper": p_hiper
            })
            if is_labeled:
                labels.append(label)

    if is_labeled:
        labels = np.array(labels)
        le = LabelEncoder()
        labels = le.fit_transform(labels)

    return path_data, labels

In [5]:
dados_anotados, rotulos = create_paths(PASTA_ANOTADAS, is_labeled=True)

## Classes de Estrator de Características e Classificadores

### Extrator de Características

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
class Extractor:
    def __init__(self, name, extractor = None, preprocess = None, rgb_only = False):
        self.extractor = extractor
        self.preprocess = preprocess
        self.path = f"D:\\Users\\Sabrina\\Documents\\trabalho_pi\\{name}"
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if extractor is not None:
            self.extractor = self.extractor.to(self.device)
            self.extractor.eval()
        else:
            self.extractor = None
        
        self.data = None
        self.rgb_only = rgb_only
    
    def extract_visual_features(self, image_np):
        img = Image.fromarray(image_np)
        tensor = self.preprocess(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feat = self.extractor(tensor)
        return feat.flatten().cpu().numpy()

    def extract_multimodal_seed_features(self, img_rgb, img_rx, cube_hiper):
        #extrando caracteristicas do RGB
        feat_rgb = self.extract_visual_features(img_rgb)
        if self.rgb_only:
            return feat_rgb
        
        #grante que o RX tem 3 canais e extrai as caracteristicas
        if len(img_rx.shape) == 2: #se for escala de cinza
            img_rx_3ch = cv2.cvtColor(img_rx, cv2.COLOR_GRAY2RGB)
        else:
            img_rx_3ch = img_rx
        feat_rx = self.extract_visual_features(img_rx_3ch)
        
        #extrai a assinatura espectral média do cubo: 150 características)
        #cube_hiper tem formato (150, altura, largura)
        feat_hiper = np.mean(cube_hiper, axis=(1, 2)) 
        self.len_hiper = feat_hiper.shape[0]
        
        #concatena todos os vetores em um só
        vetor_final = np.concatenate((feat_rgb, feat_rx, feat_hiper))
        
        return vetor_final
    
    def load_and_extract_matrix(self, path_list):
        X_caracteristicas = []
        
        for i, caminhos in enumerate(path_list):
            #ler RGB e converter para o padrão RGB (OpenCV lê em BGR)
            img_bgr = cv2.imread(caminhos["rgb"])
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            #ler Raio-X
            img_rx = cv2.imread(caminhos["rx"], cv2.IMREAD_GRAYSCALE)
            
            #ler cubo hiperespectral (.npy)
            cubo_hiper = np.load(caminhos["hiper"])
            
            vetor_semente = self.extract_multimodal_seed_features(img_rgb, img_rx, cubo_hiper)
            
            X_caracteristicas.append(vetor_semente)
            
            if (i + 1) % 100 == 0:
                print(f"  Processadas {i + 1}/{len(path_list)} sementes...")
                
        return np.array(X_caracteristicas)
    
    def run(self, list, file_name = "data"):
        os.makedirs(self.path, exist_ok=True)

        data_path = os.path.join(self.path, f"{file_name}.npy")

        if os.path.exists(data_path):
            self.data = np.load(data_path)
            self.len_hiper = 150
        else:
            if self.extractor is None:
                print("Erro: Extrator nulo, sem resultados guardados!")
                return
            print("-> Extraindo características...")
            self.data = self.load_and_extract_matrix(list)
            np.save(data_path, self.data)

        print(f"Caracteristicas salvas com sucesso em: {self.path}")

    def slice_data(self):
        len_rgb = int(self.data.shape[1] - self.len_hiper) // 2

        rgb = self.data[:, :len_rgb]
        rx = self.data[:, len_rgb:2*len_rgb]
        hiper = self.data[:, 2*len_rgb:]

        return rgb, rx, hiper

### Classificadores

In [151]:
class Classifier:
    def __init__(self):
        self.model = None

    def eval(self, x_train, y_train, x_test, y_test):
        self.model.fit(x_train, y_train)

        y_pred = self.model.predict(x_test)
        print("\n--- RESULTADO---")
        print(classification_report(y_test, y_pred))

        self.plot_confusion_matrix(y_test, y_pred)
    
    def cross_val(self, dataset, label):
        cv = StratifiedKFold(
                    n_splits=5,
                    shuffle=True,
                    random_state=42
        )
        scores = cross_val_score(
            self.model,
            dataset,
            label,
            cv=cv,
            error_score='raise',
            scoring="accuracy"
        )
        print(f"Scores per fold: {scores}")
        print(f"Mean Accuracy: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")
    
    def plot_confusion_matrix(self, y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred)

        labels = ["Heavy", "Light", "Moderate", "No"]

        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(cmap=plt.cm.Blues)
        plt.show()

class SupportVectorMachine(Classifier):
    def __init__(self):
        self.model = SVC(kernel='linear', probability=True)
    
class RandomForest(Classifier):
    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=300, random_state=42)

class XGBoost(Classifier):
    def __init__(self):
        self.model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            random_state=42,
            tree_method="hist",
            n_estimators=80,          
            max_depth=3,              
            learning_rate=0.1,
            subsample=0.8,            
            colsample_bytree=0.8,    
            n_jobs=-1          
        )

### Funções de Avaliação

In [ ]:
def training(names_class, classifiers, data, label, titulo):
    print("\n============================================")
    print(f"==============Treinando com {titulo}=============")
    print("=============================================")

    for i in range(len(classifiers)):
        print(f"{names_class[i]}")
        classifiers[i].cross_val(data, label)
        print("-------------------------------------\n")

def eval(data_extrated, label):
    names_class = ["SVM", "Random Forest", "XGBoost"]
    classifiers = [SupportVectorMachine(), RandomForest(), XGBoost()]

    training(names_class, classifiers, data_extrated.data, label, "toda a base")

    rgb, rx, hiper = data_extrated.slice_data()

    training(names_class, classifiers, rgb, label,"RGB")

    training(names_class, classifiers, rx, label, "RX")

    training(names_class, classifiers, hiper, label, "HIPER")

    rgb_rx = np.concatenate((rgb, rx), axis=1)
    rgb_hiper = np.concatenate((rgb, hiper), axis=1)
    rx_hiper = np.concatenate((rx, hiper), axis=1)

    training(names_class, classifiers, rgb_rx, label, "RGB+RX")

    training(names_class, classifiers, rgb_hiper, label, "RGB+HIPER")

    training(names_class, classifiers, rx_hiper, label, "RX+HIPER")

## Classificação pelo Classificador Intrínseco a Arquitetura

### Funções de treinamento e validação PyTorch

In [39]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float)
        self.labels = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [40]:
args = {
    'epoch_num': 10,     # Número de épocas.
    'lr': 1e-3,           # Taxa de aprendizado.
    'weight_decay': 8e-3, # Penalidade L2 (Regularização).
    'batch_size': 20,     # Tamanho do batch.
    'device': device
}

In [41]:
def def_optimizer(model):
    return torch.optim.Adam([
        {'params': model.parameters(), 
                    'lr': args['lr'], 
                    'weight_decay': args['weight_decay']
        }
    ], lr = 0)

criterion = torch.nn.CrossEntropyLoss().to(args['device'])

In [42]:
def train(train_loader, net, epoch, optimizer):
  net.to(args['device'])
  net.train()

  start = time.time()

  epoch_loss  = []
  pred_list, rotulo_list = [], []
  for k, batch in enumerate(train_loader):
    print('\r--{0}/{1}--'.format(k, len(train_loader)), end='', flush=True)
    dado, rotulo = batch

    # Cast do dado na GPU
    dado = dado.to(args['device'])
    rotulo = rotulo.to(args['device'])

    # Forward
    ypred = net(dado)
    #print(ypred.size(), rotulo.size())
    loss = criterion(ypred, rotulo)
    #print(loss.size())
    epoch_loss.append(loss.cpu().data)

    _, pred = torch.max(ypred, axis=1)
    pred_list.append(pred.cpu().numpy())
    rotulo_list.append(rotulo.cpu().numpy())

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  epoch_loss = np.asarray(epoch_loss)
  pred_list = np.concatenate(pred_list)
  rotulo_list = np.concatenate(rotulo_list)

  acc = accuracy_score(pred_list, rotulo_list)

  # end = time.time()
  # print('\n#################### Train ####################')
  # print('Epoch %d, Loss: %.4f +/- %.4f, Acc: %.2f, Time: %.2f' % (epoch, epoch_loss.mean(), epoch_loss.std(), acc*100, end-start))

  return acc, epoch_loss.mean()


In [43]:
def validate(test_loader, net, fold):
  net.to(args['device'])
  net.eval()

  start = time.time()

  epoch_loss  = []
  pred_list, rotulo_list = [], []
  with torch.no_grad():
    for k, batch in enumerate(test_loader):

      print('\r--{0}/{1}--'.format(k, len(test_loader)), end='', flush=True)
      dado, rotulo = batch

      # Cast do dado na GPU
      dado = dado.to(args['device'])
      rotulo = rotulo.to(args['device'])

      # Forward
      ypred = net(dado)
      loss = criterion(ypred, rotulo)
      epoch_loss.append(loss.cpu().data)

      _, pred = torch.max(ypred, axis=1)
      pred_list.append(pred.cpu().numpy())
      rotulo_list.append(rotulo.cpu().numpy())

  epoch_loss = np.asarray(epoch_loss)
  pred_list = np.concatenate(pred_list)
  rotulo_list = np.concatenate(rotulo_list)

  acc = accuracy_score(pred_list, rotulo_list)

  # end = time.time()
  # print('\n********** Validate **********')
  # print('Fold %d, Loss: %.4f +/- %.4f, Acc: %.2f, Time: %.2f\n' % (fold, epoch_loss.mean(), epoch_loss.std(), acc*100, end-start))

  return acc, epoch_loss.mean()


In [91]:
def do_cross_val_torch(dataset, label, net):
    dataset = CustomDataset(data=dataset, labels=label)
    kfold = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    score_acc = []
    score_loss = []
    for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(len(dataset)), label)):

        print(f'\nFold {fold+1}')

        train_dataset = Subset(dataset, train_idx)
        val_dataset = Subset(dataset, val_idx)
        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False
        )

        optimizer = def_optimizer(net)
        for epoch in range(args['epoch_num']):
            train(train_loader, net, epoch, optimizer)

        acc, loss = validate(val_loader, net,fold)

        score_acc.append(acc)
        score_loss.append(loss)

    print(f"\nAvg acurracy: {np.mean(score_acc)}")
    print(f"Avg loss: {np.std(score_loss)}")

In [ ]:
def reiniciar_pesos(camada):
    if isinstance(camada, (torch.nn.Linear, torch.nn.Conv2d)):
        torch.nn.init.kaiming_uniform_(camada.weight, a=math.sqrt(5))
        if camada.bias is not None:
            torch.nn.init.zeros_(camada.bias)

def training_torch(classifier, data, label, titulo):
    print("\n============================================")
    print(f"==============Treinando com {titulo}=============")
    print("=============================================")
    classifier.apply(reiniciar_pesos)
    do_cross_val_torch(data, label, classifier)


def eval_torch(data_extrated, label, net, index_linear):
    classifier = copy.deepcopy(net)
    out_feat = net[index_linear].out_features
    classifier[index_linear] = torch.nn.Linear(in_features=data_extrated.data.shape[1], out_features=out_feat, bias=True) 
    training_torch(classifier, data_extrated.data, label, "toda a base")

    rgb, rx, hiper = data_extrated.slice_data()
    classifier[index_linear] = torch.nn.Linear(in_features=rgb.shape[1], out_features=out_feat, bias=True) 
    training_torch(classifier, rgb, label,"RGB")

    classifier[index_linear] = torch.nn.Linear(in_features=rx.shape[1], out_features=out_feat, bias=True)
    training_torch(classifier, rx, label, "RX")

    classifier[index_linear] = torch.nn.Linear(in_features=hiper.shape[1], out_features=out_feat, bias=True)
    training_torch(classifier, hiper, label, "HIPER")

    rgb_rx = np.concatenate((rgb, rx), axis=1)
    rgb_hiper = np.concatenate((rgb, hiper), axis=1)
    rx_hiper = np.concatenate((rx, hiper), axis=1)

    classifier[index_linear] = torch.nn.Linear(in_features=rgb_rx.shape[1], out_features=out_feat, bias=True)
    training_torch(classifier, rgb_rx, label, "RGB+RX")

    classifier[index_linear] = torch.nn.Linear(in_features=rgb_hiper.shape[1], out_features=out_feat, bias=True)
    training_torch(classifier, rgb_hiper, label, "RGB+HIPER")

    classifier[index_linear] = torch.nn.Linear(in_features=rx_hiper.shape[1], out_features=out_feat, bias=True)
    training_torch(classifier, rx_hiper, label, "RX+HIPER")

## Arquiteturas

### VGG-16

In [76]:
model_vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device)

In [82]:
model_vgg16.classifier[0] = torch.nn.Linear(in_features=1174, out_features=4096, bias=True)
model_vgg16.classifier[-1] = torch.nn.Linear(in_features=4096, out_features=4, bias=True)

In [53]:
feature_extractor_vgg16  = torch.nn.Sequential(
                        model_vgg16.features,
                        torch.nn.AdaptiveAvgPool2d(1)
                    ).to(device)

preprocess_vgg16 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
vgg16 = Extractor("vgg16", feature_extractor_vgg16, preprocess_vgg16)
vgg16.run(dados_anotados)

In [39]:
eval(vgg16, rotulos)


==============Treinando com toda a base=============
SVM
Scores per fold: [0.55045872 0.53211009 0.57798165 0.55963303 0.64220183]
Mean Accuracy: 0.572 (+/- 0.076)
-------------------------------------

Random Forest
Scores per fold: [0.64220183 0.56880734 0.60550459 0.59633028 0.59633028]
Mean Accuracy: 0.602 (+/- 0.047)
-------------------------------------


==============Treinando com RGB=============
SVM
Scores per fold: [0.58715596 0.60550459 0.5412844  0.58715596 0.55963303]
Mean Accuracy: 0.576 (+/- 0.046)
-------------------------------------

Random Forest
Scores per fold: [0.57798165 0.55963303 0.56880734 0.57798165 0.55045872]
Mean Accuracy: 0.567 (+/- 0.021)
-------------------------------------


==============Treinando com RX=============
SVM
Scores per fold: [0.4587156  0.44036697 0.49541284 0.48623853 0.48623853]
Mean Accuracy: 0.473 (+/- 0.041)
-------------------------------------

Random Forest
Scores per fold: [0.4587156  0.47706422 0.51376147 0.50458716 0.5137614

In [ ]:
do_cross_val_torch(vgg16.data, rotulos, model_vgg16.classifier)

Fold 1
--13/14--
#################### Train ####################
Epoch 0, Loss: 0.9612 +/- 0.2464, Acc: 58.26, Time: 0.34
--13/14--
#################### Train ####################
Epoch 1, Loss: 0.9274 +/- 0.1815, Acc: 56.19, Time: 0.22
--13/14--
#################### Train ####################
Epoch 2, Loss: 0.7896 +/- 0.0867, Acc: 62.84, Time: 0.22
--13/14--
#################### Train ####################
Epoch 3, Loss: 0.8480 +/- 0.1684, Acc: 60.55, Time: 0.22
--13/14--
#################### Train ####################
Epoch 4, Loss: 0.8048 +/- 0.1037, Acc: 63.99, Time: 0.22
--13/14--
#################### Train ####################
Epoch 5, Loss: 0.8242 +/- 0.1605, Acc: 60.55, Time: 0.22
--13/14--
#################### Train ####################
Epoch 6, Loss: 0.7815 +/- 0.0993, Acc: 66.51, Time: 0.22
--13/14--
#################### Train ####################
Epoch 7, Loss: 0.8455 +/- 0.1417, Acc: 59.86, Time: 0.22
--13/14--
#################### Train ####################
Epoch 8, Loss: 

In [ ]:
eval_torch(vgg16, rotulos, model_vgg16.classifier, 0)


==============Treinando com toda a base=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----Avg acurracy: 0.6128440366972476
Avg loss: 0.061710044741630554

==============Treinando com RGB=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----Avg acurracy: 0.6128440366972476
Avg loss: 0.049240030348300934

==============Treinando com RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----Avg acurracy: 0.5137614678899082
Avg loss: 0.049002643674612045

==============Treinando com HIPER=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----Avg acurracy: 0.3706422018348624
Avg loss: 0.02618938498198986

==============Treinando com RGB+RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----Avg acurracy: 0.5944954128440367
Avg loss: 0.1277063637971878

==============T

### VGG-19

In [93]:
model_vgg19 = models.vgg19(weights=models.VGG19_Weights.DEFAULT).to(device)

In [94]:
model_vgg19.classifier[0] = torch.nn.Linear(in_features=1174, out_features=4096, bias=True)
model_vgg19.classifier[-1] = torch.nn.Linear(in_features=4096, out_features=4, bias=True)

In [95]:
feature_extractor_vgg19  = torch.nn.Sequential(
                                model_vgg19.features,
                                torch.nn.AdaptiveAvgPool2d(1)
                            ).to(device)

# Transformação padrão para as imagens que vão para a CNN
preprocess_vgg19 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
vgg19 = Extractor("vgg19", feature_extractor_vgg19, preprocess_vgg19)
vgg19.run(dados_anotados)

In [43]:
eval(vgg19, rotulos)


==============Treinando com toda a base=============
SVM
Scores per fold: [0.6146789  0.58715596 0.60550459 0.55963303 0.6146789 ]
Mean Accuracy: 0.596 (+/- 0.042)
-------------------------------------

Random Forest
Scores per fold: [0.66972477 0.58715596 0.62385321 0.59633028 0.6146789 ]
Mean Accuracy: 0.618 (+/- 0.058)
-------------------------------------


==============Treinando com RGB=============
SVM
Scores per fold: [0.63302752 0.55045872 0.55963303 0.59633028 0.65137615]
Mean Accuracy: 0.598 (+/- 0.079)
-------------------------------------

Random Forest
Scores per fold: [0.63302752 0.60550459 0.60550459 0.60550459 0.62385321]
Mean Accuracy: 0.615 (+/- 0.023)
-------------------------------------


==============Treinando com RX=============
SVM
Scores per fold: [0.49541284 0.47706422 0.53211009 0.52293578 0.48623853]
Mean Accuracy: 0.503 (+/- 0.042)
-------------------------------------

Random Forest
Scores per fold: [0.47706422 0.51376147 0.44036697 0.50458716 0.4862385

In [119]:
do_cross_val_torch(vgg19.data, rotulos, model_vgg19.classifier)

Fold 1
--13/14--
#################### Train ####################
Epoch 0, Loss: 1.3708 +/- 0.0640, Acc: 30.73, Time: 0.47
--13/14--
#################### Train ####################
Epoch 1, Loss: 1.1786 +/- 0.1156, Acc: 42.66, Time: 0.22
--13/14--
#################### Train ####################
Epoch 2, Loss: 1.0429 +/- 0.1463, Acc: 50.23, Time: 0.22
--13/14--
#################### Train ####################
Epoch 3, Loss: 0.9599 +/- 0.0675, Acc: 50.92, Time: 0.22
--13/14--
#################### Train ####################
Epoch 4, Loss: 0.9262 +/- 0.1454, Acc: 52.52, Time: 0.22
--13/14--
#################### Train ####################
Epoch 5, Loss: 0.9651 +/- 0.1448, Acc: 55.73, Time: 0.22
--13/14--
#################### Train ####################
Epoch 6, Loss: 0.9525 +/- 0.0975, Acc: 51.61, Time: 0.22
--13/14--
#################### Train ####################
Epoch 7, Loss: 0.9058 +/- 0.1248, Acc: 55.05, Time: 0.22
--13/14--
#################### Train ####################
Epoch 8, Loss: 

In [ ]:
eval_torch(vgg19, rotulos, model_vgg19.classifier, 0)


==============Treinando com toda a base=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.563302752293578
Avg loss: 0.12580366432666779

==============Treinando com RGB=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.6275229357798165
Avg loss: 0.12051163613796234

==============Treinando com RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.5009174311926605
Avg loss: 0.05367765948176384

==============Treinando com HIPER=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.3431192660550459
Avg loss: 0.033218950033187866

==============Treinando com RGB+RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.634862385321101
Avg loss: 0.08335258811712265

=============

### ResNet 50

In [119]:
model_resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

In [130]:
class_resnet = torch.nn.Sequential(
    torch.nn.Linear(in_features=2048, out_features=4, bias=True)
)

In [121]:
feature_extractor_resnet = torch.nn.Sequential(*list(model_resnet.children())[:-1]).to(device)

preprocess_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
resnet = Extractor("resnet50", feature_extractor_resnet, preprocess_resnet)
resnet.run(dados_anotados)

In [126]:
eval(resnet, rotulos)


==============Treinando com toda a base=============
SVM
Scores per fold: [0.57798165 0.55963303 0.55045872 0.55963303 0.58715596]
Mean Accuracy: 0.567 (+/- 0.027)
-------------------------------------

Random Forest
Scores per fold: [0.55963303 0.60550459 0.59633028 0.52293578 0.66055046]
Mean Accuracy: 0.589 (+/- 0.092)
-------------------------------------


==============Treinando com RGB=============
SVM
Scores per fold: [0.55963303 0.51376147 0.5412844  0.5412844  0.5412844 ]
Mean Accuracy: 0.539 (+/- 0.029)
-------------------------------------

Random Forest
Scores per fold: [0.60550459 0.6146789  0.51376147 0.56880734 0.58715596]
Mean Accuracy: 0.578 (+/- 0.072)
-------------------------------------


==============Treinando com RX=============
SVM
Scores per fold: [0.50458716 0.52293578 0.47706422 0.44954128 0.46788991]
Mean Accuracy: 0.484 (+/- 0.052)
-------------------------------------

Random Forest
Scores per fold: [0.52293578 0.51376147 0.4587156  0.5412844  0.5045871

In [ ]:
eval_torch(resnet, rotulos, class_resnet, 1)


==============Treinando com toda a base=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.7045871559633028
Avg loss: 0.12645414471626282

==============Treinando com RGB=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.6862385321100917
Avg loss: 0.0956563949584961

==============Treinando com RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.5688073394495412
Avg loss: 0.08735903352499008

==============Treinando com HIPER=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.36880733944954136
Avg loss: 0.02267497591674328

==============Treinando com RGB+RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.7155963302752294
Avg loss: 0.11949465423822403

============

### Inception v3

In [132]:
model_inception = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT).to(device)

In [131]:
class_inception =  torch.nn.Sequential(
    torch.nn.Dropout(p=0.5, inplace=False),
    torch.nn.Linear(in_features=2048, out_features=1000, bias=True)
)

In [133]:
model_inception.aux_logits = False
model_inception.fc = torch.nn.Identity()

feature_extractor_inception = model_inception

preprocess_inception = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
inception = Extractor("inception_v3", feature_extractor_inception, preprocess_inception)
inception.run(dados_anotados)

In [129]:
eval(inception, rotulos)


==============Treinando com toda a base=============
SVM
Scores per fold: [0.50458716 0.53211009 0.55963303 0.56880734 0.59633028]
Mean Accuracy: 0.552 (+/- 0.063)
-------------------------------------

Random Forest
Scores per fold: [0.58715596 0.60550459 0.60550459 0.63302752 0.62385321]
Mean Accuracy: 0.611 (+/- 0.032)
-------------------------------------


==============Treinando com RGB=============
SVM
Scores per fold: [0.52293578 0.49541284 0.55963303 0.55045872 0.55963303]
Mean Accuracy: 0.538 (+/- 0.050)
-------------------------------------

Random Forest
Scores per fold: [0.55963303 0.56880734 0.56880734 0.55963303 0.62385321]
Mean Accuracy: 0.576 (+/- 0.048)
-------------------------------------


==============Treinando com RX=============
SVM
Scores per fold: [0.4587156  0.41284404 0.48623853 0.4587156  0.51376147]
Mean Accuracy: 0.466 (+/- 0.067)
-------------------------------------

Random Forest
Scores per fold: [0.47706422 0.46788991 0.46788991 0.53211009 0.5688073

In [135]:
eval_torch(inception, rotulos, class_inception, 1)


==============Treinando com toda a base=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.7027522935779815
Avg loss: 0.11006276309490204

==============Treinando com RGB=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.6458715596330276
Avg loss: 0.08586666733026505

==============Treinando com RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.5761467889908257
Avg loss: 0.0716588944196701

==============Treinando com HIPER=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.3339449541284404
Avg loss: 0.31818732619285583

==============Treinando com RGB+RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.6623853211009174
Avg loss: 0.10735436528921127

=============

### EfficientNet B0

In [136]:
model_efficient = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).to(device)

In [137]:
model_efficient.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [138]:
feature_extractor_efficientnet = torch.nn.Sequential(
    model_efficient.features,
    torch.nn.AdaptiveAvgPool2d(1)
)
preprocess_efficientnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
efficientnet = Extractor("efficientnet", feature_extractor_efficientnet, preprocess_efficientnet)
efficientnet.run(dados_anotados)

In [ ]:
eval(efficientnet, rotulos)

In [140]:
eval_torch(efficientnet, rotulos, model_efficient.classifier, 1)


==============Treinando com toda a base=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.6972477064220184
Avg loss: 0.10873447358608246

==============Treinando com RGB=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.655045871559633
Avg loss: 0.10194534063339233

==============Treinando com RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.5376146788990825
Avg loss: 0.07126560062170029

==============Treinando com HIPER=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.36697247706422015
Avg loss: 0.3155752122402191

==============Treinando com RGB+RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.7082568807339449
Avg loss: 0.11860208958387375

=============

### MobileNet v3 Large

In [142]:
model_mobile = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT).to(device)

In [143]:
model_mobile.classifier

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=1000, bias=True)
)

In [144]:
feature_extractor_mobilenet = torch.nn.Sequential(
    model_mobile.features,
    torch.nn.AdaptiveAvgPool2d(1)
)
preprocess_mobilenet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
mobilenet = Extractor("mobilenet", feature_extractor_mobilenet, preprocess_mobilenet)
mobilenet.run(dados_anotados)

In [147]:
eval_torch(mobilenet, rotulos, model_mobile.classifier,0)


==============Treinando com toda a base=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.5889908256880734
Avg loss: 0.061074238270521164

==============Treinando com RGB=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.6238532110091743
Avg loss: 0.052894797176122665

==============Treinando com RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.46972477064220186
Avg loss: 0.04947035759687424

==============Treinando com HIPER=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.37431192660550455
Avg loss: 0.04282251372933388

==============Treinando com RGB+RX=============

Fold 1
--3/4----
Fold 2
--3/4----
Fold 3
--3/4----
Fold 4
--3/4----
Fold 5
--3/4----
Avg acurracy: 0.5963302752293578
Avg loss: 0.05666913837194443

========